# Data Acquisition: USGS ComCat Earthquake Catalog

This notebook fetches **25 years of earthquake data** from the
[USGS ComCat FDSN Event Web Service](https://earthquake.usgs.gov/fdsnws/event/1/)
for four study regions, then cleans and exports the catalogs as Parquet files
for downstream analysis.

**Regions covered:**

| Region | Bounding Box | Min Magnitude | Year Range |
|--------|-------------|---------------|------------|
| Global | (none) | M2.5+ | 2000–2025 |
| Oklahoma | 33.5–37.5°N, 100.0–94.5°W | M1.0+ | 2000–2025 |
| Permian Basin | 30.5–33.5°N, 105.0–100.5°W | M1.0+ | 2010–2025 |
| Southern California | 32.0–36.5°N, 121.0–114.5°W | M1.0+ | 2000–2025 |

**Outputs:**
- `data/raw/<region>/` — monthly CSV files from ComCat
- `data/processed/<region>.parquet` — cleaned, deduplicated catalogs
- `figures/magnitude_distributions.png` — summary validation figure

---
## 1. Imports & Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import src modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from src.fetch import fetch_all_regions, REGIONS
from src.catalog import (
    load_raw_csvs,
    clean_catalog,
    deduplicate,
    estimate_mc,
    REGION_COLORS,
)

# Plotting defaults
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

---
## 2. Region Configuration

The four study regions are defined in `src/fetch.py`. Below we display their
configuration for reference.

In [ ]:
# Build a summary table of region configurations
rows = []
for name, cfg in REGIONS.items():
    lat = f"{cfg['lat_range'][0]}–{cfg['lat_range'][1]}°N" if cfg["lat_range"] else "(global)"
    lon = f"{cfg['lon_range'][0]}–{cfg['lon_range'][1]}°E" if cfg["lon_range"] else "(global)"
    years = f"{cfg['years'].start}–{cfg['years'].stop - 1}"
    rows.append({
        "Region": name,
        "Latitude": lat,
        "Longitude": lon,
        "Min Magnitude": cfg["min_mag"],
        "Year Range": years,
    })

config_df = pd.DataFrame(rows)
config_df

---
## 3. Fetch Data from USGS ComCat

> **Note:** This step takes approximately **30–60 minutes** depending on
> network speed and API load. The fetcher saves each month as an individual
> CSV file under `data/raw/<region>/`, so if the process is interrupted it
> will **resume from where it left off** on the next run. You can safely
> re-run this cell without re-downloading already-fetched months.

In [ ]:
# Fetch all four regions (with resume support)
raw_data = fetch_all_regions(base_dir=RAW_DIR)

In [ ]:
# Quick summary of raw download counts
for region_name, df in raw_data.items():
    print(f"{region_name:25s} {len(df):>8,} raw events")

---
## 4. Data Cleaning

For each region we:

1. Load raw CSVs from `data/raw/<region>/`
2. Filter to rows where `type == "earthquake"` (removing blasts, quarry events, etc.)
3. Remove events with NaN magnitudes
4. Parse `time` column to proper datetime
5. Deduplicate events reported by multiple networks using `src.catalog.deduplicate()`
6. Report counts before and after cleaning

In [ ]:
# Map from fetch region names to catalog region names
REGION_MAP = {
    "global": "global",
    "oklahoma": "oklahoma",
    "permian_basin": "permian",
    "southern_california": "socal",
}

cleaned = {}

for fetch_name, catalog_name in REGION_MAP.items():
    raw_dir = RAW_DIR / fetch_name
    
    print(f"\n{'='*50}")
    print(f"Region: {fetch_name}")
    print(f"{'='*50}")
    
    # Load raw CSVs
    df_raw = load_raw_csvs(raw_dir)
    n_raw = len(df_raw)
    print(f"  Raw events loaded:       {n_raw:>8,}")
    
    # Filter to earthquake type
    if "type" in df_raw.columns:
        n_non_eq = (df_raw["type"].str.lower().str.strip() != "earthquake").sum()
        print(f"  Non-earthquake events:   {n_non_eq:>8,}")
    
    # Clean: filter type, parse time, drop NaN magnitudes
    df_clean = clean_catalog(df_raw)
    n_clean = len(df_clean)
    print(f"  After type/NaN filter:   {n_clean:>8,}")
    
    # Deduplicate
    df_dedup = deduplicate(df_clean)
    n_dedup = len(df_dedup)
    n_dupes = n_clean - n_dedup
    print(f"  Duplicates removed:      {n_dupes:>8,}")
    print(f"  Final event count:       {n_dedup:>8,}")
    
    cleaned[catalog_name] = df_dedup

---
## 5. Save Processed Data

Each cleaned catalog is saved as a Parquet file under `data/processed/`.

In [ ]:
for catalog_name, df in cleaned.items():
    out_path = PROCESSED_DIR / f"{catalog_name}.parquet"
    df.to_parquet(out_path, index=False)
    size_mb = out_path.stat().st_size / 1e6
    print(f"  Saved {catalog_name:20s} -> {out_path.name}  ({len(df):>8,} events, {size_mb:.1f} MB)")

print("\nAll processed catalogs saved.")

---
## 6. Validation & Summary

In [ ]:
# Print event counts per region
print("Event counts per region (cleaned & deduplicated):")
print("-" * 45)
for catalog_name, df in cleaned.items():
    tmin = df["time"].min().strftime("%Y-%m-%d")
    tmax = df["time"].max().strftime("%Y-%m-%d")
    print(f"  {catalog_name:20s} {len(df):>8,} events  ({tmin} to {tmax})")

In [ ]:
# Color mapping for plots
COLORS = {
    "oklahoma": "#E63946",
    "permian": "#F4A261",
    "socal": "#457B9D",
    "global": "#2A9D8F",
}

LABELS = {
    "oklahoma": "Oklahoma",
    "permian": "Permian Basin",
    "socal": "Southern California",
    "global": "Global (M2.5+)",
}

In [ ]:
# Magnitude distribution histograms (4 subplots)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Magnitude Distributions by Region", fontsize=14, fontweight="bold")

for ax, (catalog_name, df) in zip(axes.flat, cleaned.items()):
    color = COLORS[catalog_name]
    label = LABELS[catalog_name]
    
    ax.hist(
        df["mag"],
        bins=80,
        color=color,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.3,
    )
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Magnitude")
    ax.set_ylabel("Count")
    ax.set_yscale("log")
    
    # Mark estimated Mc
    mc = estimate_mc(df["mag"])
    ax.axvline(mc, color="k", linestyle="--", linewidth=1, label=f"Mc = {mc:.1f}")
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "magnitude_distributions.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'magnitude_distributions.png'}")

In [ ]:
# Event counts per year (time series)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Earthquake Counts per Year by Region", fontsize=14, fontweight="bold")

for ax, (catalog_name, df) in zip(axes.flat, cleaned.items()):
    color = COLORS[catalog_name]
    label = LABELS[catalog_name]
    
    yearly = df.groupby(df["time"].dt.year).size()
    ax.bar(yearly.index, yearly.values, color=color, alpha=0.85, edgecolor="white", linewidth=0.3)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Year")
    ax.set_ylabel("Events")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "events_per_year.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'events_per_year.png'}")

In [ ]:
# Estimate Mc (magnitude of completeness) for each region
print("Estimated magnitude of completeness (Mc) per region:")
print("-" * 45)
for catalog_name, df in cleaned.items():
    mc = estimate_mc(df["mag"], method="maxc")
    print(f"  {LABELS[catalog_name]:25s}  Mc = {mc:.1f}")

---
## 7. Data Quality Notes

### Known issues and caveats

- **Oklahoma network expansion (~2010):** The Oklahoma Geological Survey
  significantly expanded its seismic monitoring network around 2010 in response
  to the sharp increase in induced seismicity. Events before this expansion have
  a higher magnitude of completeness (Mc), meaning small earthquakes were
  under-reported prior to ~2010. Temporal analyses should account for this
  change in detection capability.

- **Magnitude type mixing:** The USGS catalog reports magnitudes using several
  scales (Ml, Mw, Md, mb, etc.). The `mag` column is the "preferred" magnitude
  chosen by the reporting network, which may use different scales for different
  events. For small regional events, Ml (local magnitude) is most common,
  while global events tend to use Mw (moment magnitude). The `magType` column
  can be used for scale-specific filtering if needed.

- **Permian Basin coverage (pre-2010):** The TexNet seismic network was not
  deployed until 2017, so Permian Basin coverage before that date relies on
  sparser regional and national networks. We start the Permian Basin catalog
  at 2010 to partially mitigate this, but completeness still improves
  substantially after 2017.

- **Deduplication tolerances:** Events within 2 seconds and 0.05° (~5.5 km)
  in latitude/longitude are considered potential duplicates. This is
  conservative and may occasionally merge distinct events from swarm
  sequences. The deduplication keeps the report with the highest station
  count (`nst`).

- **API rate limiting:** The USGS ComCat API has a 20,000-event limit per
  query. Our monthly pagination keeps individual requests well below this
  threshold for all regions and magnitude ranges used here.